# **Хрупкость карт значимости и защиты: атака, сдвиг входа, β-сглаживание, сертификат**

Практика к модулю [«Атаки на объяснения»](https://ai-interpretability.school/learn/22), уроки «Хрупкость карт значимости», «Геометрия и манипуляция моделью» и «Защиты и сертификация».

В уроках карта значимости ломалась четырьмя способами, и от двух из них предлагались защиты. Здесь все это считается на одной маленькой сверточной сети для MNIST, которую мы обучим за пару минут:

1. **атака Ghorbani, Abid и Zou** ([AAAI 2019](https://arxiv.org/abs/1710.10547)): возмущение входа в шаре $\|\delta\|_\infty\le\varepsilon$ уводит карту от исходной, а предсказанный класс не меняется; точка сравнения — случайное возмущение того же бюджета;
2. **проверка input invariance Kindermans и соавторов** ([2017](https://arxiv.org/abs/1711.00867)): вход сдвигают на константу, смещение первого слоя этот сдвиг компенсирует, и мы смотрим, какие методы `captum.attr` отвечают на сдвиг, на который модель не отвечает;
3. **целевая манипуляция Dombrowski и соавторов** ([NeurIPS 2019](https://arxiv.org/abs/1906.07983)): карта одного изображения переводится в карту другого при почти тех же выходах сети;
4. **β-сглаживание** из той же работы: карта считается на сети, где ReLU заменены на $\mathrm{softplus}_\beta$;
5. **softplus и weight decay при обучении** ([Dombrowski и соавторы, 2022](https://arxiv.org/abs/2012.10425)): сеть с самого начала обучают с гладкой активацией и штрафом на веса;
6. **Sparsified SmoothGrad Levine, Singla и Feizi** ([NeurIPS 2019](https://arxiv.org/abs/1905.12105)): сглаженная разреженная карта и сертификат — нижняя граница числа пикселей top-$K$, которые останутся в top-$K$ при любом возмущении радиуса $\rho$.

Устойчивость карт меряем своими функциями (пересечение top-$k$, корреляции Спирмена и Пирсона) и двумя метриками библиотеки Quantus: `quantus.MaxSensitivity` и `quantus.LocalLipschitzEstimate`.

Все случайное зафиксировано сидом 0. Обучение сети и итерации атак зависят от железа и версий библиотек, поэтому числа ниже у вас могут отличаться в последних знаках; выводы от этого не меняются.

In [ ]:
!pip install -q captum==0.9.0 quantus==0.6.0

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # Captum и Quantus предупреждают о хуках и о формате входа; tqdm — об ipywidgets
import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import quantus
from scipy.stats import spearmanr
from torch.distributions import Normal
from torchvision import datasets, transforms
from captum.attr import Saliency, InputXGradient, IntegratedGradients, NoiseTunnel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
np.random.seed(0)
print('device:', device)

## 1. Данные и сеть

MNIST грузится из `torchvision` без ключей. Пиксели в $[0,1]$, без нормировки: в разделе 3 мы сами сдвинем вход на константу.

Сеть маленькая: две свертки, усредняющий пулинг и два полносвязных слоя. Два выбора в ней сделаны ради проверок ниже:

- **у первой свертки нет паддинга.** Паддинг дописывает нули по краю, а нуль после сдвига входа уже не соответствует нулю до сдвига, и компенсация Kindermans и соавторов на краях перестает быть точной;
- **пулинг усредняющий, а не max.** На черном фоне MNIST у max-пулинга много равных значений, и какое из них выиграет, решает округление в последнем знаке. Тогда даже градиент двух математически одинаковых сетей разойдется, и проверка начнет мерить округление, а не метод.

Функция `make_net(act)` принимает класс активации: так та же архитектура собирается и с ReLU, и с softplus.

In [ ]:
to_tensor = transforms.ToTensor()
train_set = datasets.MNIST('data', train=True, download=True, transform=to_tensor)
test_set = datasets.MNIST('data', train=False, download=True, transform=to_tensor)
X_test = test_set.data[:2000].float().div(255).unsqueeze(1).to(device)
Y_test = test_set.targets[:2000].to(device)


def make_net(act):
    return nn.Sequential(nn.Conv2d(1, 16, 5), act(), nn.Conv2d(16, 32, 5), act(), nn.AvgPool2d(2),
                         nn.Flatten(), nn.Linear(32 * 10 * 10, 64), act(), nn.Linear(64, 10)).to(device)


def train(net, weight_decay=0.0, epochs=2, seed=0):
    torch.manual_seed(seed)
    loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,
                                         generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=weight_decay)
    net.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            F.cross_entropy(net(xb.to(device)), yb.to(device)).backward()
            opt.step()
    return net.eval()


def accuracy(net):
    with torch.no_grad():
        return (net(X_test).argmax(1) == Y_test).float().mean().item()


t0 = time.time()
torch.manual_seed(0)
net = train(make_net(nn.ReLU))
print(f'ReLU-сеть: точность на 2000 тестовых изображениях {accuracy(net):.4f}, обучение {time.time() - t0:.0f} с')

Общие функции тетради:

- `saliency_map(model, x, c, create_graph)` — карта простого градиента, как у Ghorbani и соавторов: модуль градиента оценки класса $c$ по входу, нормированный так, что сумма по пикселям равна единице. С `create_graph=True` карта остается дифференцируемой по входу, и по ней можно вести атаку;
- `smooth_copy(model, beta)` — копия сети с теми же весами, где каждый `nn.ReLU` заменен на `nn.Softplus(beta)`;
- `topk_overlap(a, b, k)` — доля общих пикселей среди $k$ самых важных у двух карт;
- `spearman(a, b)` и `pearson(a, b)` — корреляции двух карт для каждой строки батча.

Атакуем и проверяем 50 первых тестовых изображений, которые сеть классифицирует правильно.

In [ ]:
def saliency_map(model, x, c, create_graph=False):
    x = x if x.requires_grad else x.clone().requires_grad_(True)
    score = model(x)[torch.arange(len(x)), c].sum()
    g, = torch.autograd.grad(score, x, create_graph=create_graph)
    a = g.abs().flatten(1)
    return a / a.sum(1, keepdim=True)


def smooth_copy(model, beta):
    m = copy.deepcopy(model)
    for i, layer in enumerate(m):
        if isinstance(layer, nn.ReLU):
            m[i] = nn.Softplus(beta)
    return m.eval()


def topk_overlap(a, b, k):
    ia = a.topk(k, 1).indices
    ib = b.topk(k, 1).indices
    hit = torch.zeros_like(a, dtype=torch.bool).scatter_(1, ia, True).gather(1, ib)
    return hit.float().sum(1) / k


def spearman(a, b):
    a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    return np.array([spearmanr(p, q).statistic for p, q in zip(a, b)])


def pearson(a, b):
    a = a - a.mean(1, keepdim=True)
    b = b - b.mean(1, keepdim=True)
    return (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1))


with torch.no_grad():
    pred = net(X_test).argmax(1)
correct = torch.nonzero(pred == Y_test).flatten()
x0, c0 = X_test[correct[:50]], pred[correct[:50]]
K_TOP = 78                                  # 10 % из 784 пикселей
h0 = saliency_map(net, x0, c0)
print('батч:', tuple(x0.shape), 'классы:', c0[:10].tolist(), '...')

## 2. Атака Ghorbani, Abid и Zou

### 2.1. Почему атаке нужен softplus

Top-k атака максимизирует $D(x)=-\sum_{i\in B}I(x)_i$: суммарную важность, которую карта $I(x)$ отдает пикселям $B$, самым важным на исходной карте, атака снижает. Шаг атаки идет по знаку $\nabla_x D$, а $D$ сама построена из градиента. Значит, нужна вторая производная сети по входу.

**Задание 1. Посчитайте $\nabla_x D$ для первого изображения батча дважды: через ReLU-сеть `net` и через ее копию `smooth_copy(net, 5.0)`. Множество $B$ — `K_TOP` самых важных пикселей исходной карты ReLU-сети. Сколько элементов градиента из 784 отличны от нуля в каждом случае?**

In [ ]:
# Ваш код здесь

У ReLU-сети градиент $D$ равен нулю тождественно, и это не округление. Внутри каждой области, где знаки входов ReLU не меняются, сеть — линейная функция входа: свертки, усредняющий пулинг и полносвязные слои линейны, а ReLU на такой области — умножение на постоянную маску. Градиент линейной функции от входа не зависит, его производная по входу — ноль. Поэтому Ghorbani и соавторы, а за ними Dombrowski и соавторы, ведут атаку по softplus-копии сети, а готовое возмущение проверяют на исходной ReLU-сети.

### 2.2. Top-k атака

**Задание 2. Напишите функцию `ghorbani_topk(model, x, c, eps, beta=5.0, steps=50)`. Она делает `steps` шагов $x\leftarrow\Pi\big(x+\alpha\,\mathrm{sign}\,\nabla_x D\big)$ с $\alpha=\varepsilon/10$, где градиент $D$ считается по `smooth_copy(model, beta)`, а $\Pi$ — проекция в шар $\|x-x_t\|_\infty\le\varepsilon$ и в диапазон пикселей $[0,1]$. После каждого шага $D$ пересчитывается на исходной `model`; функция возвращает для каждого изображения то возмущенное изображение, у которого $D$ на исходной сети наибольшая при прежнем классе.**

Подсказка: если класс сменился на всех шагах, верните исходное изображение — тогда атака для него просто не удалась.

In [ ]:
# Ваш код здесь

Сравним атаку с точкой сравнения Ghorbani и соавторов — случайным возмущением того же бюджета: каждый пиксель сдвигается на $\pm\varepsilon$ со случайным знаком. Для каждого $\varepsilon$ печатаем долю изображений с прежним классом, среднее пересечение top-78 исходной и новой карты и среднюю корреляцию Спирмена между ними.

In [ ]:
rows, attacked = [], {}
for eps in (0.05, 0.1, 0.2):
    gen = torch.Generator().manual_seed(0)
    x_rand = (x0 + eps * (torch.randint(0, 2, x0.shape, generator=gen).to(device) * 2 - 1)).clamp(0, 1)
    attacked[eps] = ghorbani_topk(net, x0, c0, eps)
    for kind, xx in (('случайное ±ε', x_rand), ('top-k атака', attacked[eps])):
        h1 = saliency_map(net, xx, c0)
        with torch.no_grad():
            same = (net(xx).argmax(1) == c0).float().mean().item()
        rows.append({'ε': eps, 'возмущение': kind, 'класс прежний': same,
                     'пересечение top-78': topk_overlap(h0, h1, K_TOP).mean().item(),
                     'Спирмен': spearman(h0, h1).mean()})
table_attack = pd.DataFrame(rows)
print(table_attack.round(3).to_string(index=False))

In [ ]:
i = 0
xa = attacked[0.1][i:i + 1]
panels = ((x0[i, 0], 'исходное изображение'), (xa[0, 0], 'после атаки, ε = 0.1'),
          (h0[i].view(28, 28), 'карта исходного'), (saliency_map(net, xa, c0[i:i + 1])[0].view(28, 28), 'карта после атаки'))
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img.detach().cpu(), cmap='gray' if 'изображение' in title or 'после атаки,' in title else 'hot')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle(f'Класс {c0[i].item()} до и после атаки; пересечение top-78 карт — '
             f'{topk_overlap(h0[i:i + 1], saliency_map(net, xa, c0[i:i + 1]), K_TOP).item():.2f}', fontsize=11)
fig.tight_layout()
plt.show()

### 2.3. Гиперпараметр атаки: $\beta$ softplus-копии

$\beta$ решает, насколько softplus-копия похожа на ReLU-сеть. При малом $\beta$ копия гладкая, но ее градиент уже плохо описывает настоящую сеть; при большом $\beta$ копия почти совпадает с ReLU-сетью, а ее вторая производная сосредоточена у изломов и почти везде близка к нулю. Проверим, как от $\beta$ зависит успех атаки при $\varepsilon=0.1$.

In [ ]:
for beta in (1.0, 5.0, 20.0, 100.0):
    xb = ghorbani_topk(net, x0, c0, 0.1, beta=beta)
    print(f'β = {beta:>5}: пересечение top-78 после атаки {topk_overlap(h0, saliency_map(net, xb, c0), K_TOP).mean():.3f}')

## 3. Сдвиг входа на константу: проверка Kindermans и соавторов

Сеть `net` видит пиксели в $[0,1]$. Построим вторую сеть `net_shift`, которая видит те же изображения в кодировке $[-1,0]$, то есть $x_2=x_1+m_2$ с $m_2=-1$, и на соответствующих входах выдает те же числа. Для этого достаточно поправить смещение первой свертки: $b_2=b_1-\sum W\,m_2$, где сумма идет по входному каналу и окну ядра. Остальные веса общие.

Метод объяснения проходит проверку, если карты двух сетей на соответствующих входах совпадают. Меряем относительное расхождение $\|a_1-a_2\|_2/\|a_1\|_2$ и берем наибольшее по батчу. Порог 0.01 стоит с запасом в обе стороны: расхождение у прошедших методов — порядка ошибки округления float32, у не прошедших — порядка единицы.

In [ ]:
M2 = -1.0
net_shift = copy.deepcopy(net)
with torch.no_grad():
    net_shift[0].bias -= M2 * net_shift[0].weight.sum((1, 2, 3))
x1, x2 = x0, x0 + M2
with torch.no_grad():
    print(f'наибольшая разница выходов двух сетей: {(net(x1) - net_shift(x2)).abs().max().item():.1e}')


def rel_diff(a1, a2):
    return ((a1 - a2).flatten(1).norm(dim=1) / a1.flatten(1).norm(dim=1)).max().item()

**Задание 3. Посчитайте `rel_diff` карт двух сетей для пяти методов `captum.attr` и сведите их в таблицу «метод — расхождение — проходит ли проверку (расхождение < 0.01)». Методы: `Saliency` (с `abs=False`), `InputXGradient` (Gradient×Input), `IntegratedGradients` с нулевым опорным входом у обеих сетей, `IntegratedGradients` с опорным входом «черное изображение» (у `net` это 0, у `net_shift` это $m_2$), `NoiseTunnel(Saliency)` в режиме SmoothGrad (16 копий, `stdevs=0.15`). У IG возьмите `n_steps=64`. У SmoothGrad шум случайный: перед вызовом для каждой сети ставьте `torch.manual_seed(0)`, чтобы обе сети получили один и тот же шум. Сколько методов из пяти проходят проверку?**

In [ ]:
# Ваш код здесь

Почему Gradient×Input не проходит, видно без таблицы: градиенты двух сетей одинаковы, а умножаются они на разные входы, поэтому разность атрибуций равна $\nabla_x f\odot m_2$. Проверим это тождество и посмотрим на карты.

In [ ]:
a1, a2 = pairs['Gradient×Input']
grad = Saliency(net).attribute(x1, target=c0, abs=False)
print(f'наибольшая разница между a2 - a1 и grad * m2: {(a2 - a1 - grad * M2).abs().max().item():.1e}')

i = 0
names = ('Saliency', 'Gradient×Input', 'IG, нулевой опорный вход', 'IG, черное изображение')
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, name in enumerate(names):
    for row, (a, title) in enumerate(zip(pairs[name], ('пиксели в [0, 1]', 'пиксели в [-1, 0]'))):
        m = a[i, 0].detach().cpu()
        lim = m.abs().max()
        axes[row, col].imshow(m, cmap='bwr', vmin=-lim, vmax=lim)
        axes[row, col].set_title(f'{name}\n{title}', fontsize=9)
        axes[row, col].axis('off')
fig.tight_layout()
plt.show()

### 3.1. Где ломается сама проверка: паддинг

Компенсация через смещение точна, только если каждый выход первого слоя видит лишь настоящие пиксели. Соберем ту же первую свертку с паддингом 2 и сравним выходы двух кодировок: внутри изображения они совпадают, а у края паддинг дописывает нуль, который в кодировке $[-1,0]$ означает уже серый пиксель, а не черный.

In [ ]:
conv_pad = nn.Conv2d(1, 16, 5, padding=2).to(device)
conv_pad.load_state_dict(net[0].state_dict())
conv_pad_shift = copy.deepcopy(conv_pad)
with torch.no_grad():
    conv_pad_shift.bias -= M2 * conv_pad_shift.weight.sum((1, 2, 3))
    diff = (conv_pad(x1) - conv_pad_shift(x2)).abs().amax((0, 1))
print(f'наибольшая разница выходов внутри (строки и столбцы 2..25): {diff[2:-2, 2:-2].max().item():.1e}')
print(f'наибольшая разница выходов у края: {diff.max().item():.2f}')

## 4. Целевая манипуляция: Dombrowski и соавторы

Теперь у атаки есть цель: карта $h^t$ другого изображения. Манипулированное изображение $x_{adv}$ ищется минимизацией

$$\mathcal{L}=\frac{\|h(x_{adv})-h^t\|^2}{\|h^t\|^2}+\gamma\,\big\|p(x_{adv})-p(x)\big\|^2,$$

где $p$ — вероятности классов у ReLU-сети. Первое слагаемое тянет карту к цели (мы делим его на $\|h^t\|^2$, чтобы оно было порядка единицы), второе держит выходы на месте. У авторов во втором слагаемом стоят выходы сети, и бюджета по норме нет; мы держим вероятности и добавляем бюджет $\|x_{adv}-x\|_\infty\le\varepsilon$, иначе на MNIST оптимизация просто перерисует цифру.

Пары: первые 20 изображений батча — исходные, следующие 20 — источник целевых карт; цель $h^t$ — карта второго изображения пары для его собственного класса.

**Задание 4. Напишите функцию `dombrowski(expl_model, attack_model, x, c, h_t, eps, gamma=10.0, steps=300, lr=0.01)`. Она минимизирует $\mathcal{L}$ по $x_{adv}$ оптимизатором Adam, считая карту $h$ через `attack_model` с `create_graph=True`, а вероятности — через `net`; после каждого шага $x_{adv}$ проецируется в шар $\|x_{adv}-x\|_\infty\le\varepsilon$ и в $[0,1]$. Параметр `expl_model` нужен, чтобы атака знала, чью карту она в итоге будет обманывать; в этом разделе `attack_model = smooth_copy(net, 5.0)` и `expl_model = net`.**

In [ ]:
# Ваш код здесь

Печатаем долю пар с прежним классом, корреляцию Пирсона карты с целью до атаки и после, корреляцию новой карты с исходной и медиану изменения вероятностей $\|p(x_{adv})-p(x)\|_1$.

In [ ]:
xs, cs = x0[:20], c0[:20]
xt, ct = x0[20:40], c0[20:40]


def manipulation_row(expl_model, attack_model, eps, label):
    h_t = saliency_map(expl_model, xt, ct)
    h_s = saliency_map(expl_model, xs, cs)
    xa = dombrowski(expl_model, attack_model, xs, cs, h_t, eps)
    h_a = saliency_map(expl_model, xa, cs)
    with torch.no_grad():
        same = (net(xa).argmax(1) == cs).float().mean().item()
        dp = (F.softmax(net(xa), 1) - F.softmax(net(xs), 1)).abs().sum(1).median().item()
    return xa, {'карта': label, 'ε': eps, 'класс прежний': same,
                'Пирсон с целью до': pearson(h_s, h_t).mean().item(),
                'Пирсон с целью после': pearson(h_a, h_t).mean().item(),
                'Пирсон с исходной': pearson(h_a, h_s).mean().item(), 'медиана Δp': dp}


x_manip, row = manipulation_row(net, smooth_copy(net, 5.0), 0.2, 'ReLU')
print(pd.DataFrame([row]).round(3).to_string(index=False))

In [ ]:
i = 0
h_t = saliency_map(net, xt[i:i + 1], ct[i:i + 1])
panels = ((xs[i, 0], 'исходное', 'gray'), (x_manip[i, 0], 'манипулированное', 'gray'), (xt[i, 0], 'источник цели', 'gray'),
          (saliency_map(net, xs[i:i + 1], cs[i:i + 1])[0].view(28, 28), 'карта исходного', 'hot'),
          (saliency_map(net, x_manip[i:i + 1], cs[i:i + 1])[0].view(28, 28), 'карта манипулированного', 'hot'),
          (h_t[0].view(28, 28), 'целевая карта', 'hot'))
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img.detach().cpu(), cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
with torch.no_grad():
    fig.suptitle(f'Класс исходного {cs[i].item()}, манипулированного {net(x_manip[i:i + 1]).argmax().item()}, '
                 f'источника цели {ct[i].item()}', fontsize=11)
fig.tight_layout()
plt.show()

## 5. β-сглаживание

По теореме 1 Dombrowski и соавторов изменение карты при движении входа вдоль поверхности постоянного выхода ограничено величиной, пропорциональной $\beta$. Защита — считать карту на `smooth_copy(net, β)` с малым $\beta$. Проверим ее против атаки, которая знает о защите: атака теперь направлена прямо на сглаженную карту, то есть `expl_model = attack_model = smooth_copy(net, β)`, и градиент атаки точный.

In [ ]:
rows = [row]
for beta in (10.0, 3.0, 1.0):
    smoothed = smooth_copy(net, beta)
    rows.append(manipulation_row(smoothed, smoothed, 0.2, f'β = {beta}')[1])
table_beta = pd.DataFrame(rows)
print(table_beta.round(3).to_string(index=False))

У β-сглаживания есть оборотная сторона, о которой легко забыть: `smooth_copy(net, β)` — это другая функция, и ее карта объясняет уже не совсем ту сеть, которая делает прогноз. Посмотрим, насколько при каждом $\beta$ совпадают прогнозы копии с прогнозами сети и насколько карта копии похожа на карту ReLU-сети.

In [ ]:
with torch.no_grad():
    pred_relu = net(X_test).argmax(1)
for beta in (100.0, 10.0, 3.0, 1.0):
    smoothed = smooth_copy(net, beta)
    with torch.no_grad():
        agree = (smoothed(X_test).argmax(1) == pred_relu).float().mean().item()
    sim = pearson(saliency_map(smoothed, x0, c0), h0).mean().item()
    print(f'β = {beta:>5}: прогноз совпадает с ReLU-сетью у {agree:.3f} изображений, Пирсон карты с картой ReLU-сети {sim:.3f}')

## 6. Softplus и weight decay при обучении: Dombrowski и соавторы, 2022

β-сглаживание меняет только способ считать карту. Вторая работа Dombrowski и соавторов меняет саму модель: граница на норму гессиана выхода по входу убывает вместе с нормами весов и с $\Sigma_2$ — пределом второй производной активации. Отсюда два способа из трех: обучать сеть с softplus с самого начала и с weight decay. Третий способ, штраф за норму гессиана, требует производных третьего порядка на каждом шаге обучения, и здесь мы его не повторяем.

Обучим ту же архитектуру с `nn.Softplus(10)` и weight decay $5\cdot10^{-4}$ — это значения $\beta$ и $\lambda$ из таблицы 1 статьи.

In [ ]:
t0 = time.time()
torch.manual_seed(0)
net_soft = train(make_net(lambda: nn.Softplus(10.0)), weight_decay=5e-4)
print(f'сеть с softplus β=10 и weight decay: точность {accuracy(net_soft):.4f}, обучение {time.time() - t0:.0f} с')

Мера устойчивости из статьи — корреляция Пирсона между картой входа и картой того же входа с гауссовым шумом уровня $\nu$. Сравним три модели: ReLU-сеть, ее β-сглаженную копию с $\beta=3$ и сеть, обученную с softplus и weight decay. Шум один и тот же для всех моделей.

In [ ]:
models = {'ReLU': net, 'β-сглаживание β=3': smooth_copy(net, 3.0), 'обучена с softplus и weight decay': net_soft}
noise = torch.randn(x0.shape, generator=torch.Generator().manual_seed(0)).to(device)
rows = []
for nu in (0.05, 0.1, 0.2):
    row = {'ν': nu}
    for name, m in models.items():
        with torch.no_grad():
            c = m(x0).argmax(1)
        row[name] = pearson(saliency_map(m, x0, c), saliency_map(m, x0 + nu * noise, c)).mean().item()
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 6.1. Те же модели в метриках Quantus

`quantus.MaxSensitivity` ([Yeh и соавторы, 2019](https://arxiv.org/abs/1901.09392)) берет `nr_samples` случайных возмущений входа в шаре радиуса `lower_bound` и возвращает наибольшее относительное изменение карты; `quantus.LocalLipschitzEstimate` ([Alvarez-Melis и Jaakkola, 2018](https://arxiv.org/abs/1806.08049)) — наибольшее отношение изменения карты к изменению входа при гауссовом шуме со стандартным отклонением `perturb_std`. У обеих метрик меньше — устойчивее. Объяснение Quantus получает функцией `explain_func(model, inputs, targets)`, которая принимает и возвращает массивы NumPy.

**Задание 5. Посчитайте обе метрики для трех моделей словаря `models` на батче `x0`: `nr_samples=10`, `lower_bound=0.05` у `MaxSensitivity`, `perturb_std=0.05` у `LocalLipschitzEstimate`, объяснение — `Saliency` из Captum, класс — прогноз той же модели. Перед каждым вызовом метрики ставьте `np.random.seed(0)`: Quantus берет шум из NumPy. Выведите таблицу «модель — точность — среднее MaxSensitivity — среднее LocalLipschitzEstimate».**

In [ ]:
# Ваш код здесь

## 7. Sparsified SmoothGrad: сертификат вместо эвристики

Levine, Singla и Feizi доказывают для любой карты $h$ со значениями в $[0,1]$ и ее сглаживания $\bar h(x)=\mathbb{E}_{\epsilon\sim\mathcal N(0,\sigma^2I)}\,h(x+\epsilon)$: если

$$\Phi\Big(\Phi^{-1}\big(\bar h_{[i]}(x)\big)-\frac{2\rho}{\sigma}\Big)\ \ge\ \bar h_{[2K-i]}(x),$$

то при любом возмущении $\|\tilde x-x\|_2\le\rho$ хотя бы $i$ пикселей из top-$K$ сглаженной карты останутся в top-$K$. Здесь $\bar h_{[j]}$ — $j$-й по величине элемент, $\Phi$ — функция распределения стандартного нормального закона. Сертификат $R_{cert}$ — наибольшее такое $i$.

Две базовые карты:

- **SmoothGrad**: модуль градиента каждой зашумленной копии, деленный на свой максимум, чтобы значения легли в $[0,1]$;
- **Sparsified SmoothGrad**: у каждой зашумленной копии доля $\tau$ самых больших по модулю элементов градиента получает 1, остальные 0.

Сглаженную карту оцениваем средним по $q$ копиям. Строгий сертификат требует доверительных границ для каждого $\bar h_{[j]}$; сначала считаем сертификат по точечным оценкам, а строгий вариант — в конце раздела. Берем $K=196$, четверть пикселей, как в эксперименте авторов на MNIST.

**Задание 6. Напишите две функции. `smooth_map(model, x, c, sigma, q, tau=None, seed=0)` возвращает сглаженную карту формы `(len(x), 784)`: при `tau=None` — SmoothGrad с нормировкой каждой копии на ее максимум, иначе Sparsified SmoothGrad с долей `tau` (число единиц — `int(tau * 784)`). Шум берите из `torch.Generator().manual_seed(seed)`; копии удобно считать пачками по 128. `certificate(h, K, rho, sigma)` возвращает для каждой строки `h` наибольшее $i\le K$, при котором выполнено условие теоремы, или 0.**

In [ ]:
# Ваш код здесь

Сертифицированная доля top-$K$, $R_{cert}/K$, в среднем по 10 изображениям для трех уровней шума $\sigma$ и трех радиусов $\rho$; $q=256$ копий, $\tau=0.1$.

In [ ]:
K_CERT = 196
xc, cc = x0[:10], c0[:10]
rows, maps = [], {}
for sigma in (0.25, 0.5, 1.0):
    maps[sigma] = (smooth_map(net, xc, cc, sigma, 256), smooth_map(net, xc, cc, sigma, 256, tau=0.1))
    for rho in (0.1, 0.25, 0.5):
        rows.append({'σ': sigma, 'ρ': rho,
                     'SmoothGrad': certificate(maps[sigma][0], K_CERT, rho, sigma).float().mean().item() / K_CERT,
                     'Sparsified SmoothGrad': certificate(maps[sigma][1], K_CERT, rho, sigma).float().mean().item() / K_CERT})
table_cert = pd.DataFrame(rows)
print(table_cert.round(3).to_string(index=False))

Сертификат — нижняя граница, и ее можно сравнить с тем, что происходит на деле. Возьмем $\sigma=0.5$, $\rho=0.25$ и для каждого из 10 изображений 20 случайных возмущений с $\|\delta\|_2=\rho$; посчитаем Sparsified SmoothGrad возмущенного входа с другим сидом шума и наименьшую по возмущениям долю общих пикселей top-$K$. Случайное возмущение — не наихудшее, поэтому наблюдаемая доля обязана быть не ниже сертификата, но может быть выше: сертификат обещает только то, что выдержит любое возмущение.

In [ ]:
sigma, rho = 0.5, 0.25
h_clean = maps[sigma][1]
cert = certificate(h_clean, K_CERT, rho, sigma).float() / K_CERT
gen = torch.Generator().manual_seed(1)
worst = torch.ones(len(xc), device=device)
for r in range(20):
    d = torch.randn(xc.shape, generator=gen).to(device)
    d = rho * d / d.flatten(1).norm(dim=1).view(-1, 1, 1, 1)
    h_pert = smooth_map(net, xc + d, cc, sigma, 256, tau=0.1, seed=100 + r)
    worst = torch.min(worst, topk_overlap(h_clean, h_pert, K_CERT))
print('сертификат R_cert / K:      ', np.round(cert.numpy(), 2))
print('наблюдаемый минимум по 20:  ', np.round(worst.cpu().numpy(), 2))

### 7.1. Строгий вариант: доверительные границы

Каждое $\bar h_j$ — среднее $q$ величин из $[0,1]$, и по неравенству Хеффдинга с вероятностью не меньше $1-\alpha$ сразу для всех 784 пикселей оценка отличается от настоящего значения не больше чем на $e=\sqrt{\ln(2\cdot784/\alpha)/(2q)}$. Строгий сертификат подставляет в левую часть условия нижнюю границу, $\bar h_{[i]}-e$, а в правую — верхнюю, $\bar h_{[2K-i]}+e$. Посмотрим, сколько от сертификата остается при $\alpha=0.05$ и разном $q$.

In [ ]:
def certificate_strict(h, K, rho, sigma, q, alpha=0.05):
    e = np.sqrt(np.log(2 * h.shape[1] / alpha) / (2 * q))
    normal = Normal(0.0, 1.0)
    hs = h.sort(1, descending=True).values.double().cpu()
    i = torch.arange(1, K + 1)
    lhs = normal.cdf(normal.icdf((hs[:, :K] - e).clamp(1e-12, 1 - 1e-12)) - 2 * rho / sigma)
    ok = lhs >= hs[:, 2 * K - i - 1] + e
    return torch.where(ok.any(1), (ok * i).amax(1), torch.zeros(len(h), dtype=torch.long)), e


for q in (256, 1024, 4096):
    h_q = smooth_map(net, xc, cc, 0.5, q, tau=0.1)
    strict, e = certificate_strict(h_q, K_CERT, 0.25, 0.5, q)
    point = certificate(h_q, K_CERT, 0.25, 0.5)
    print(f'q = {q:>4}: e = {e:.3f}, сертификат по точечным оценкам {point.float().mean() / K_CERT:.3f}, '
          f'строгий {strict.float().mean() / K_CERT:.3f}')

**Задание 7. По таблицам разделов 5–7: какая из защит дает гарантию для конкретного входа и радиуса, а какие только снижают чувствительность карты на опыте? Что каждая из них меняет — способ считать карту или саму модель — и чем за это платят (точность, совпадение с объясняемой сетью, число проходов)?**

**Ваш ответ:**

## Что унести из тетради

- **Неизменный прогноз не гарантирует неизменной карты.** Top-k атака Ghorbani и соавторов в шаре $L_\infty$ сдвигает карту сильнее случайного возмущения того же бюджета, а класс не меняется ни у одного изображения. Точка сравнения обязательна: часть расхождения дает уже случайный шум.
- **У ReLU-сети вторая производная по входу равна нулю тождественно**, поэтому атаку ведут по softplus-копии, а проверяют на исходной сети. От $\beta$ этой копии зависит, насколько атака удается.
- **Сдвиг входа на константу с компенсацией в смещении модель не меняет, а Gradient×Input и IG с нулевым опорным входом меняет.** Простой градиент, SmoothGrad над ним и IG с опорным входом, который сдвигается вместе с данными, проверку проходят. Сама проверка требует, чтобы первый слой не видел паддинга.
- **Целевая манипуляция Dombrowski и соавторов подтягивает карту к карте другого изображения** при почти тех же вероятностях классов.
- **β-сглаживание и обучение с softplus и weight decay снижают чувствительность карты, но ничего не гарантируют.** β-сглаживание к тому же объясняет другую функцию: при малом $\beta$ прогнозы копии расходятся с прогнозами сети.
- **Sparsified SmoothGrad дает сертификат**, и он содержательнее сертификата обычного SmoothGrad; растет с $\sigma$ и убывает с $\rho$. Строгий сертификат требует доверительных границ, а с ними — тысяч зашумленных копий на один вход.